# 📊 เมทริกซ์ความสับสน (Confusion Matrix)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **เมทริกซ์ความสับสน (Confusion Matrix)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างผลลัพธ์การทำนายจำลองสำหรับปัญหาการจำแนก 3 คลาส (`flange` แปลนท่อ, `valve` วาล์ว, `gauge` เกจวัด) เพื่อสังเกตจุดที่โมเดลเกิดความสับสนคลาส
2. เขียนสคริปต์คำนวณ Confusion Matrix ทั้งแบบดิบและแบบปรับขนาด (Normalized) จากศูนย์ด้วย NumPy
3. ตรวจสอบความถูกต้องเปรียบเทียบกับไลบรารีมาตรฐาน `scikit-learn`
4. พล็อตกราฟแผนภาพความร้อน (Heatmaps) เพื่อชี้เป้าความผิดพลาดการทำงานของแบบจำลองได้อย่างชัดเจนด้วยสายตา
5. อธิบายโครงสร้างของ Confusion Matrix ของ YOLO สำหรับหลายคลาส รวมถึงวิธีการจำแนกกลุ่มข้อมูลที่เป็น **ฉากหลัง (Background)**

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลจำลอง (Data Generation)

เราจะทำการสร้างข้อมูลจำลองของป้ายกำกับเฉลัยจริงและการทำนายของอุปกรณ์จำนวน 150 ชิ้น:
*   คลาส 0: `flange` (แปลนท่อ)
*   คลาส 1: `valve` (วาล์ว)
*   คลาส 2: `gauge` (เกจวัด)

สมมติให้แบบจำลองมีประสิทธิภาพดีมากในการจำแนกประเภทแปลนท่อ แต่มีแนวโน้มที่จะทายเกจวัดผิดสับสนเป็นวาล์วอยู่บ่อยครั้ง

In [ ]:
# สร้างคลาสเฉลยจริงในสัดส่วนที่เท่ากัน (อย่างละ 50 ชิ้น)
y_true = np.concatenate([np.zeros(50), np.ones(50), np.ones(50)*2]).astype(int)

# จำลองผลการทำนายที่มีความสับสนสลับคลาสตามสมมติฐาน
y_pred = y_true.copy()

# แฟลนจ์/แปลนท่อ (คลาส 0): ทายถูก 90%, ทายสับสนเป็นวาล์ว 10%
confuse_0 = np.random.choice(50, 5, replace=False)
y_pred[confuse_0] = 1

# วาล์ว (คลาส 1): ทายถูก 80%, ทายสับสนเป็นแปลนท่อ 10%, ทายสับสนเป็นเกจวัด 10%
confuse_1_to_0 = np.random.choice(50, 5, replace=False) + 50
confuse_1_to_2 = np.random.choice(50, 5, replace=False) + 50
y_pred[confuse_1_to_0] = 0
y_pred[confuse_1_to_2] = 2

# เกจวัด (คลาส 2): ทายถูก 60%, ทายสับสนเป็นวาล์ว 30%, ทายสับสนเป็นแปลนท่อ 10%
confuse_2_to_1 = np.random.choice(50, 15, replace=False) + 100
confuse_2_to_0 = np.random.choice(50, 5, replace=False) + 100
y_pred[confuse_2_to_1] = 1
y_pred[confuse_2_to_0] = 0

## 2. การสร้าง Confusion Matrix จากพื้นฐาน (Calculating the Confusion Matrix from Scratch)

เราจะพัฒนาฟังก์ชันเพื่อคำนวณหาตารางเมทริกซ์ความสับสนขนาด $3\times3$:
-   **แกนนอน (Rows):** แสดงประเภทคลาสเป้าหมายที่เป็นเฉลยจริง (Actual Class)
-   **แกนตั้ง (Columns):** แสดงประเภทคลาสที่แบบจำลองทำนาย (Predicted Class)
-   สมาชิก $C_{i,j}$ ในตาราง จะเป็นตัวนับว่าข้อมูลเฉลยจริงคลาส $i$ ถูกทายสับสนเป็นคลาส $j$ ไปกี่จุดตัวอย่าง

In [ ]:
def custom_confusion_matrix(y_true, y_pred, n_classes=3):
    """
    สร้างตารางเมทริกซ์ความสับสนแบบดิบจากศูนย์
    """
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def custom_normalized_confusion_matrix(cm):
    """
    ทำ Normalize ตารางเมทริกซ์ความสับสนด้วยผลรวมของแต่ละแถว (ตามปริมาณคลาสเฉลยจริง)
    """
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return cm.astype(float) / row_sums

# รันฟังก์ชันคำนวณที่เราเขียนเอง
cm_raw = custom_confusion_matrix(y_true, y_pred, n_classes=3)
cm_norm = custom_normalized_confusion_matrix(cm_raw)

# เรียกคำนวณผ่าน scikit-learn เพื่อเปรียบเทียบผล
cm_sklearn = confusion_matrix(y_true, y_pred)

print("--- Custom Raw Matrix ---")
print(cm_raw)
print("\n--- Sklearn Raw Matrix ---")
print(cm_sklearn)
print("\nVerify identical matches:", np.array_equal(cm_raw, cm_sklearn))

print("\n--- Custom Normalized Matrix ---")
print(np.round(cm_norm, 4))

## 3. การพล็อตแสดงแผนภาพเมทริกซ์ความสับสน (Confusion Matrix Visualization)

เราลองมาพล็อตตารางผลลัพธ์ทั้งแบบดิบและแบบปรับสเกลเคียงคู่กันผ่านแผนภูมิความร้อน (Heatmaps)

In [ ]:
class_names = ['flange', 'valve', 'gauge']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# พล็อตตารางความสับสนดิบ
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Raw Confusion Matrix")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("Actual Class")

# พล็อตตารางความสับสนหลัง Normalize
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Normalized Confusion Matrix")
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("Actual Class")

plt.tight_layout()
plt.show()

สังเกตว่าแนวเส้นทแยงมุมของตารางหลังการทำ Normalize จะบ่งชี้ถึงค่าความถูกต้องเฉพาะกลุ่มคลาส (Recall accuracies) ได้แก่:
*   ค่าความแม่นยำของคลาสแปลนท่อ: 90%
*   ค่าความแม่นยำของคลาสวาล์ว: 80%
*   ค่าความแม่นยำของคลาสเกจวัด: 60%
นอกจากนี้ในช่องแถวของ `gauge` และคอลัมน์ของ `valve` แสดงตัวเลข `0.30` ซึ่งชี้ชัดเจนว่าโมเดลสับสนเกจวัดเป็นวาล์วสูงถึง 30%

## 💡 ความเชื่อมโยงสู่ Computer Vision และ YOLO
ในงานตรวจจับวัตถุ YOLO มีวิธีสร้างตาราง Confusion Matrix อย่างไร เนื่องจากไม่มีการนับจุดตัวอย่างข้อมูลเดี่ยวๆ ที่ฟิกซ์แน่นอนเหมือนงานแยกประเภทภาพปกติ?
คำตอบคือ YOLO ได้ขยายขอบเขตการคำนวณตารางเมทริกซ์ความสับสนโดยการเพิ่มคอลัมน์และแถวพิเศษที่ชื่อว่า **Background (ฉากหลัง)** เข้าไป:
1.  **การคาดการณ์หลุดเป้า (Prediction Mismatch):** หากกรอบเฉลยจริงคลาส `control-valve` ไม่มีกล่องทำนายใดของโมเดลวาดพิกัดล้อมครอบได้เกินเกณฑ์ $IoU \ge 0.45$ เลย YOLO จะถือว่าโมเดล **ทำวัตถุหลุดหาย (Missed Object)** และจะทำการบวกคะแนนความผิดพลาดนี้เข้าไปที่ แถว `control-valve` คอลัมน์ `Background` (ซึ่งคือค่าลวงลบ False Negative)
2.  **สัญญาณเตือนภัยลวง (False Alarm):** หากโมเดลตรวจจับวาดกรอบสีน้ำเงินคลาส `flange` ขึ้นมาบนพื้นที่รอยต่อภาพถ่ายที่ไม่มีวัตถุเป้าหมายใดอยู่ตรงนั้นเลย จะนับเป็น **การเข้าใจผิดว่าฉากหลังเป็นวัตถุ (False Alarm)** และจะบวกคะแนนความผิดนี้เข้าไปที่ แถว `Background` คอลัมน์ `flange` (ซึ่งคือค่าลวงบวก False Positive)
การดัดแปลงนี้ทำให้ YOLO สามารถวิเคราะห์จุดบกพร่องด้านการจำแนกชนิดคลาสวัตถุ ควบคู่ไปกับความคลาดเคลื่อนของการลากขอบเขตกล่องระบุตำแหน่ง (เช่น พลาดวัตถุเป้าหมาย หรือการวาดกล่องมั่วในพื้นที่รกร้าง) ได้อย่างสมบูรณ์แบบ